# non-diff-fn-wrap composite — cx18: wrap argmax with is_differentiable=False; output is a graph leaf

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `non-diff-fn-wrap`, `is-differentiable-flag`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "non-diff-fn-wrap"
DD_ATOM_IDS = ["non-diff-fn-wrap", "is-differentiable-flag"]
DD_SUBTOPICS = ["Backprop: non-differentiable fn wrap", "Backprop: is_differentiable flag"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Composing non-diff-fn-wrap with the is_differentiable flag

Some forward ops have no useful gradient — `t.argmax` returns int64 indices, `t.eq` returns bools. We want to call them on `MiniTensor`s without:
  - setting `requires_grad=True` on the output (no gradient could flow back).
  - attaching a `Recipe` (the reverse pass would try to recurse past it and either crash or waste work).

The fix is the `is_differentiable=False` kwarg on `wrap_forward_fn`, captured in the wrapper's closure. When the flag is False, the wrapper:
  1. Still computes the forward result (we DO want the value).
  2. Forces `requires_grad=False` and `recipe=None` on the output.
  3. Makes the output behave like a graph leaf — backprop's sorted-graph walk stops there naturally.

### Composite Exercise — wrap argmax with is_differentiable=False; output is a graph leaf

**Atoms exercised together**: `non-diff-fn-wrap`, `is-differentiable-flag`

Implement `cx18_wrap_forward_fn(fwd_fn, is_differentiable=True)`. Behavior:

1. Unbox each `MiniTensor` arg to its `.array`; pass non-Tensors through.
2. Call `fwd_fn(*raw_args, **kwargs)`.
3. Three-gate AND: `requires_grad = grad_tracking_enabled AND is_differentiable AND any(input is tracked MiniTensor)`. Read `grad_tracking_enabled` from `globals()` FRESH each call.
4. Build `out = MiniTensor(out_raw, requires_grad=rg)`. Only when `rg` is True, attach `out.recipe = Recipe(...)`. Otherwise leave `recipe=None`.

Then wrap `t.argmax` with `is_differentiable=False` and prove the output is a graph leaf.

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

from dataclasses import dataclass, field
from typing import Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    def __init__(self, array, requires_grad=False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe

def cx18_wrap_forward_fn(fwd_fn, is_differentiable=True):
    raise NotImplementedError

def cx18_is_leaf(node):
    """True iff this MiniTensor terminates the backward graph (recipe is None)."""
    raise NotImplementedError

def _test_cx18():
    globals()['grad_tracking_enabled'] = True

    add    = cx18_wrap_forward_fn(t.add)
    argmax = cx18_wrap_forward_fn(t.argmax, is_differentiable=False)
    eq     = cx18_wrap_forward_fn(t.eq,     is_differentiable=False)

    x = MiniTensor(t.tensor([3.0, 1.0, 4.0, 1.0, 5.0, 9.0, 2.0]), requires_grad=True)
    y = MiniTensor(t.tensor([1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0]), requires_grad=True)

    # (a) diff op: requires_grad propagates, Recipe is built
    out_add = add(x, y)
    assert out_add.requires_grad is True
    assert out_add.recipe is not None and out_add.recipe.func is t.add
    assert cx18_is_leaf(out_add) is False, 'differentiable add must NOT be a leaf'

    # (b) argmax: forward value still correct, but output is a graph leaf
    out_ax = argmax(x)
    assert out_ax.array.item() == 5, f'forward value wrong: {out_ax.array}'
    assert out_ax.requires_grad is False, 'non-diff must force requires_grad=False'
    assert out_ax.recipe is None, 'non-diff must NOT build a Recipe'
    assert cx18_is_leaf(out_ax) is True, 'argmax output must be a graph leaf'

    # (c) argmax dtype is int64 — proves we really called torch.argmax, not a stand-in
    assert out_ax.array.dtype in (t.int64, t.long), f'argmax should return int64: {out_ax.array.dtype}'

    # (d) eq: bool dtype, still a leaf
    out_eq = eq(x, y)
    assert out_eq.array.dtype == t.bool
    assert out_eq.requires_grad is False and out_eq.recipe is None
    assert cx18_is_leaf(out_eq) is True

    # (e) is_differentiable is STICKY (closure capture). Many calls, still leaf.
    for _ in range(3):
        assert cx18_is_leaf(argmax(x)) is True

    # (f) global toggle off → add ALSO becomes a leaf (gate 1 fails),
    #     but argmax stays a leaf for an INDEPENDENT reason (gate 2)
    globals()['grad_tracking_enabled'] = False
    try:
        assert cx18_is_leaf(add(x, y)) is True, 'toggle-off makes diff op a leaf too'
        assert cx18_is_leaf(argmax(x)) is True, 'non-diff still a leaf'
    finally:
        globals()['grad_tracking_enabled'] = True

    # (g) chained call: add(argmax(x).float(), x) — but argmax's output stays a leaf
    # i.e. backprop would stop at out_ax even if a downstream op tracks through it.
    out_ax2 = argmax(x)
    assert out_ax2.recipe is None  # graph terminates here, period.
    _dd_passed.add('cx18')

_test_cx18()

<details><summary>Show solution — cx18</summary>

```python
def cx18_wrap_forward_fn(fwd_fn, is_differentiable=True):
    def tensor_func(*args, **kwargs):
        raw_args = tuple(
            a.array if isinstance(a, MiniTensor) else a for a in args
        )
        out_raw = fwd_fn(*raw_args, **kwargs)
        rg = (
            globals()['grad_tracking_enabled']
            and is_differentiable
            and any(
                isinstance(a, MiniTensor) and a.requires_grad for a in args
            )
        )
        out = MiniTensor(out_raw, requires_grad=rg)
        if rg:
            parents = {
                i: a for i, a in enumerate(args) if isinstance(a, MiniTensor)
            }
            out.recipe = Recipe(fwd_fn, raw_args, kwargs, parents)
        return out
    return tensor_func

def cx18_is_leaf(node):
    return node.recipe is None
```

`recipe=None` is what makes the node terminal. The sorted-graph walk in `backprop` checks `if node.recipe is None: continue` — that's the exact line that protects the reverse pass from trying to differentiate argmax. The is_differentiable closure is the input contract; recipe=None is the output contract.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx18'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx18',
        'subtopics': ["Backprop: non-differentiable fn wrap", "Backprop: is_differentiable flag"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()